# P06 metadata join and unannotated review export
Uses a pinned official MetaTool commit. Run in Google Colab; upload `source_queries_for_review.csv`. No annotation labels are generated.


In [ ]:
# P06 richer-metadata join and review inventory (Google Colab)
# Generated as a reproducibility aid. Not executed by ChatGPT or on Colab.
import pandas as pd, json, subprocess, pathlib, hashlib
from google.colab import files
repo = pathlib.Path('/content/MetaTool')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/HowieHwong/MetaTool.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','35e81bb7576826e980c80fed8f8c0a2b4a1e6fbb'],check=True)
print('SOURCE COMMIT', subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())
uploaded=files.upload()
source=pd.read_csv('source_queries_for_review.csv').fillna('')
with open(repo/'dataset/plugin_info.json',encoding='utf-8') as f: info=json.load(f)
with open(repo/'dataset/plugin_des.json',encoding='utf-8') as f: des=json.load(f)
info_df=pd.DataFrame(info)
print('plugin_info entries',len(info_df),'unique model ids',info_df.name_for_model.nunique())
print('duplicated ids',info_df[info_df.name_for_model.duplicated(keep=False)].name_for_model.unique().tolist())
meta=(info_df.groupby('name_for_model',as_index=False).agg({'name_for_human':'first','description_for_model':'first','description_for_human':'first'}))
join=source.merge(meta,left_on='Tool',right_on='name_for_model',how='left',validate='many_to_one')
join['rich_metadata_exact_match']=join['name_for_model'].notna()
join.to_csv('P06_50_Queries_With_Rich_Metadata.csv',index=False)
mapping=pd.DataFrame({'tool_id':list(des.keys()),'short_description':list(des.values())})
mapping=mapping.merge(meta,left_on='tool_id',right_on='name_for_model',how='left',validate='one_to_one')
mapping['rich_metadata_exact_match']=mapping['name_for_model'].notna()
mapping.to_csv('P06_All_199_Metadata_Mapping.csv',index=False)
print('exact mapped tools',int(mapping.rich_metadata_exact_match.sum()),'of',len(mapping))
print('matched sampled queries',int(join.rich_metadata_exact_match.sum()),'of',len(join))
# Fixed, explicit candidate inventories are necessary BEFORE any three-way label.
# An inventory of all 199 tool IDs is NOT proof that a query is SELECT/NO_VALID_TOOL/ASK.
# Human reviewers must verify availability, schema evidence and all valid tool alternatives.
review=join.loc[join.rich_metadata_exact_match,['case_id','Query','Tool','original_tool_description','description_for_model']].copy()
for col in ['inventory_tool_ids','mandatory_user_requirements','supporting_source_evidence','candidate_valid_tool_ids','reviewer_A','reviewer_B','adjudicated_action','reason_for_exclusion']:
    review[col]=''
review.to_csv('P06_Review_Packet_Unannotated.csv',index=False)
for path in ['P06_50_Queries_With_Rich_Metadata.csv','P06_All_199_Metadata_Mapping.csv','P06_Review_Packet_Unannotated.csv']:
    print('CREATED',path, hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest())
    files.download(path)

